# Investigating the impact of the leakage from the Amsterdam Rijnkanaal

Low lying polders adjacent to the Amsterdam Rijnkanaal (ARK) experience more and more
leakage probably from the nearby ARK, which has a water level several meters higher
than that of the polders. The question is what causes this increase of the leagkage occurring
over the last two decades and how can it be solved by use of a layar of a sand-bentione mixture
at the canal bottom.

To investigate this, we'll generate cross section and model them in detail. With this or these
models, different impacts can be simulated and the impact possible measures can be examined for
their effectiveness.

The work is done for Rijkswaterstaat.

@ TO 2026-04-03

In [ ]:
import os
import sys
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pdf2image
import pickle

from mf6lab.Projects.ARK_RWS.src import ARK_fdm

from mf6lab.Projects.ARK_RWS.src.ARK_fdm import (
    CrossSectionDigitizer,
    ImagePicker,
    Dirs,
    plot_result,
    show_filled_array,
    litho_classes,
    geo_units
    )

from tools.fdm.src.mfgrid import Grid

print(sys.executable)

# --- Needed to make figure separate from the notebook and interactive
%matplotlib qt

# --- Seet the namespace for the relevant directories
dirs = Dirs()

WHITE = np.array([1., 1., 1.])

loading mfpath.py
/Users/Theo/Development/python/mf6_tools/mf6lab/.venv/bin/python


# Get the geotop.pdf obtained from dinoloket.nl

When reading it with pdf2image you get actually two pdfs.
One has the actual cross section and the other the legend and a map.

In [3]:
# --- Show images one after the other

for geotop_pdf in glob(dirs.dino + '*.pdf'):
    basename = os.path.basename(geotop_pdf)
        
    # --- The dpi=300 determines the pixel size of the image !
    # --- Results in one PIL object for each page in the pdf file.
    geotop1, geotop2 = pdf2image.convert_from_path(geotop_pdf, dpi=300)
    
    # --- Convert to RGB (each color as int 0-255)
    geotop1 = np.asarray(geotop1.convert("RGB"))  
    geotop2 = np.asarray(geotop2.convert("RGB"))
    
    # --- Imshow uses 0-255 as color if dtype is ints and 0-1. when dtype is float
    # --- Just to show it, doesn't matter.
    
    # --- The first pdf page contains the x-section
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle(basename)
    ax.set_title("Geotop Cross Section")
    ax.imshow(geotop1) 
    ax.plot()
    
    # --- The second pdf page contains the legend and a small map.
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle(basename)
    ax.set_title("Geotop, legend and map")
    ax.imshow(geotop2)
    ax.plot()

    

# Example (using the last geotop pdf file in dirs.dino)

Get the last pdf file in the dirs.dino directory.
Show its properties.

## Get the file its, contents and show its properties

In [2]:
print("Specifications of the Geotop pdf")
print("================================")
print("Directory dirs.dino:")
print(dirs.dino)
print()
# --- Get the last geotop.pdf file in dirs.dino
geotop_pdf = glob(dirs.dino + '*.pdf')[-1]

print("Filename:")
print(os.path.basename(geotop_pdf))
print()

print("File contents:")
print("==============")
# --- Convert the pdf to image. Each page in the pdf becomes a PIL object
# --- Note that the dpi argument determines the pixel size of the image
geotop_pages = pdf2image.convert_from_path(geotop_pdf, dpi=300)
print(f"Number of page in geotop pdf file: {len(geotop_pages)}")

# --- Get the two pages
geotop1, geotop2 = geotop_pages

# --- Show their types
print(f"Type geotop1: {type(geotop1)}")
print(f"Type geotop2: {type(geotop2)}")

# --- Convert the PIL ismage to RGB
geotop1 = np.asarray(geotop1.convert("RGB"))  
geotop2 = np.asarray(geotop2.convert("RGB"))

# --- Show their shape
print(f"geotop1 shape = {geotop1.shape}")
print(f"geotop2.shape = {geotop2.shape}")

# --- Show their RGB dtype and range
print(f"geotop1.dtype = {geotop1.dtype}, RGB values: {geotop1.min()}-{geotop1.max()}")
print(f"geotop2.dtype = {geotop2.dtype}, RGB values: {geotop2.min()}-{geotop2.max()}")


Specifications of the Geotop pdf
Directory dirs.dino:
/Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/dinoloket/

Filename:
BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130173,479431.pdf

File contents:
Number of page in geotop pdf file: 2
Type geotop1: <class 'PIL.PpmImagePlugin.PpmImageFile'>
Type geotop2: <class 'PIL.PpmImagePlugin.PpmImageFile'>
geotop1 shape = (2481, 3508, 3)
geotop2.shape = (2481, 3508, 3)
geotop1.dtype = uint8, RGB values: 0-255
geotop2.dtype = uint8, RGB values: 0-255


## Hand-written geoCodes and world-extent for this cross section

We will combine the legend colors with the hand written geoCodes pertaining to this
cross section. They are paired.

And later on with the hand-written world_extent of this cross section. The vertical
z is in m relatieve to NAP (National Datum) while the x-coordinate is in m relative
to the left-most point of the cross section. Notice that the actual national
coordinates of this starting point is in the file name and that a small
map showing where the cross section is, is in the second page of the geotop's pdf file.

In [3]:
geoCodes = ['NUECga', 'NUECgb', 'NUEC1', 'NUNIHO', 'NUNIBA', 'NUBXWI-SI-KO', 'NUBX', 'NUDR', 'NUgs']

# --- extent is always (xmin, xmax, zmin zmax). In this case in m and z (vertical)
world_extent=(0, 8625, -48.5, 0)

## Use the image picker with the legend image to get the legend colors

Instantiate the ImagePicker with the image that holds the map with the legend color boxes.
Then zoom in and click the legend's color boxes in the normal order.
This yields the colors to compare those sampled in the actual cross section with.

Press ENTER to finish with the legend.

In [4]:
# ---Instantiate the picker with the image to pick from (image with the legend)
picker = ImagePicker(geotop2)

# --- Zoom before clicking (zoom into legend). The easiest way is the shift the figure
# --- to hit the top panel of the screen. The shifted window will go full screen automatically.

# --- Step 1: Pick legend colors one after the other in sequence. Press ENTER when done.
colors = picker.get_colors(n=-1)

# --- Special for legend colors Remove white points first to remove mistake clicks
legend_colors = np.array([clr for clr in colors if not np.all(clr > 0.95)])

# --- Internally set the legend_colors (obtained from clicking the legend)
legend_colors = np.vstack((np.array([WHITE]), legend_colors))
geoCodes.insert(0, 'none')


# --- Show the legend_color geoCode pairs for this cross section.
print("Legend_color     geoCode")
print("============     =======")
for l_color, g_code in zip(legend_colors, geoCodes):
    print(np.round(l_color, 2), g_code)


Legend_color     geoCode
============     =======
[1. 1. 1.] none
[0.78 0.78 0.78] NUECga
[0.62 0.31 0.25] NUECgb
[0.   0.57 0.  ] NUEC1
[0.76 0.81 0.36] NUNIHO
[1. 1. 0.] NUNIBA
[0.95 0.88 0.02] NUBXWI-SI-KO
[0.91 0.76 0.09] NUBX
[0.85 0.64 0.13] NUDR
[0.37 0.37 1.  ] NUgs


## With the legend_colors now obtained get the pixel extent of the cross section

To get the pixel extent of the cross section:

1. Instantiate the ImagePicker once again, but now with the image of the actual cross section.
2. Zoom in the same way has before. Preferably by shifting the image window to the top of the screen to let it go full screen.
3. Then pick the corners of the cross sections of which you know the world coordinates.
4. Press ENTER when finished. The pixel_exent is thus obtained.

In fact, a pixel bounding box is computed around all picked points and turned into the
pixel_extent: (pxmin, pxmax, pymin, pymax)


In [5]:
# --- Initiate a new picker, now with the cross section   
# --- To get the pixel bounding box
picker = ImagePicker(geotop1)

# --- Again, zoom in
# --- In fact you can click any number of points, the bbox uses there min and max coords
pxl_extent = picker.get_pxl_bbox(n=-1)

print("Pixel extent of the cross section:")
print("==================================")
print("[pxmin pxmax pymin pymax]")
print(np.array(pxl_extent))

Points picked in pixels:  [(368, 397), (2964, 1705)]
Pixel extent of the cross section:
[pxmin pxmax pymin pymax]
[ 368 2964  397 1705]


## Sample the actual cross section automatically

Sampling the actual cross section image for to match each
sampled point with the legend is done automatically based on

1. The legend colors.
2. The pixel extent.
3. The world extent.
4. The voxel size if the geotop image (dx, dz), generally (100, 0.5)

The subdivision of the grid is based on the world_extent and the voxel size dx, dz.

To match empyt voxels:
1. 'none' is prepended to the geoCodes
2. WHITE = np.array([1., 1., 1.]) is prepenced to the legend_colors

The voxel array will be filled with legend codes.
Code 0 indicates empty (no legend) and get legend 'none' if required.

In [6]:
# --- Fill an array with soil indices where each index is the number of the
# --- legend color boxes in that order (the order clicked before)

# --- Instantiate the CrossSectionDigitizer using the cross section.
digitizer = CrossSectionDigitizer(image=geotop1)

# --- Set pxl_extent. We just obtained it
digitizer.set_pxl_bbox(*pxl_extent)

# --- Set world extent (world_extent was given above)
digitizer.set_world_bbox(*world_extent)

digitizer.legend_colors = legend_colors

# --- Set voxel size.
digitizer.set_grid(dx=100, dz=0.5)

# --- Fill an array of a cross section shape obtaine from dx, dz and world_extent
# --- This is done inside build
arr = digitizer.build_array()

## Show he legend indices of the sampled the cross section

The digitizer's build_array contains the legend indices for each voxel.
Empty voxels have index 0. The largest index is the index of the last legend
index (but with 'none' as the first).

The sampling was done af follows:
1. The color values of a small patch around the centre of each voxel are used.
2. Dark colors are removed to get rid of text (using brightness computation).
3. The median of the colors is taken.
4. The distance in RGB from this color to the legend_colors is computed
5. The nearest legend_color's index is found and filled into the arr array.

The resulting voxel lhrnf index array can subsequently be converted to
any property that can be linked to the legend. The best way to do this is by
using a pandas DataFrame or a dictionary with the same index as the legend colorboxes.

The underlying grid is that defined by the world_extent and the (dx,dz) voxel size.
So arr.shape == geoImage.shape(:2). The latter has RGB values and is 3D.

An mfgrid.Grid object can be used as an alternative grid, but this is not
recommended because it can create artefacts: When the resolution is too fine
these artefacts are caused by lines and text in the image.

Resampling on other grids should be done separately.

In [ ]:
# --- Show the cross section using imshow, which fills the voxels
plot_result(arr, world_extent)

# --- Add title and save
fig = plt.gcf()
fig.suptitle(f"""{os.path.basename(geotop_pdf)}
                with colors converted to soil-indices
                """)

# --- Save the image for later reporting
fig.savefig(os.path.join(dirs.images, f"{os.path.basename(geotop_pdf)}"))

plt.show()

This finishes the work flow. But with a 

# Now for all cross sections at once (in a loop)

## Get the legend colors of each geotop X-section

The legend is the second page of each geotop pdf file. Each
image is loaded in turn and the colors of the legend are clicked
and stored in a dictionary whose keyse are the basename of the
mentioned files.

These colors should be matched with the labels of the legends.
But these labels have to be provided by the user and are
given below for the current geotop pdfs


In [ ]:
# --- Coordinate data for the various cross sections

# --- Dictionary to store the picked points (pixels)
geotopLegends = {}

# --- Run over all the geotop files stored.
for geotop_pdf in glob(dirs.dino + '*.pdf'):
    
    # --- Use  basename as key.
    basename = os.path.basename(geotop_pdf)
    
    # --- Get the two pages of each geotop pdf as RGB image 
    geotop_xsec, geotop_leg = pdf2image.convert_from_path(geotop_pdf, dpi=300)
        
    geotop_xsec = np.asarray(geotop_xsec.convert("RGB"))  # map
    geotop_leg  = np.asarray(geotop_leg.convert("RGB"))  # legend (not used here) 
    
    # --- Instantiate the point picker with the map image
    picker = ImagePicker(geotop_leg)
    
    # --- Click the 5 points
    legend_colors = picker.get_colors(n=-1)

    # --- Store them
    geotopLegends[basename] = legend_colors
    
    

Value(False)


ModeResult(mode=np.int64(1), count=np.int64(1))

In [115]:
labels = ["NUAAOP	NUECgb	NUEC1	NUNIHO	NUNIBA	NUNBXWI-SI-KO	NUBX	NUKR-BXDE	NUDR	Nugs	NUUR2	NUST",
"a	v	k	kz	zf	zm	zg	g	she",
"a	v	k	kz	zf	zm	zg	g	she",		
"NUECga	NUECgb	NUEC1	NUNIHO	NUNIBA	NUNBXWI-SI-KO	NUBX	NUKR-BXDE	NUDR	NUgs	NUUR2	NUST",
"NUAAOP	NUECga	NUECgb	NUEC1	NUNIHO	NUNIBA	NUNBXWI-SI-KO	NUBX	NUDR	NUgs	NUST",
"NUAAOP	NUECga	NUECgb	NUEC1	NUNIHO	NUNIBA	NUNBXWI-SI-KO	NUBX	NUKR-BXDE	NUDR	NUgs	NUUR2	NUST",
"NUAAOP	NUECgb	NUEC1	NUNIHO	NUNIBA	NUNBXWI-SI-KO	NUBX	NUKR-BXDE	NUDR	NUgs	NUUR2	NUST",
"a	v	k	kz	zf	zm	zg	g	she",
"a	v	k	kz	zf	zm	zg	g	she",
"a	v	k	kz	zf	zm	zg	g	she"
]

geolegs = {}

for (fname, colors), label in zip(geotopLegends.items(), labels):    

    # --- Put the empty cell in front
    clrs = list(colors)    
    clrs = [colors[-1]] + colors[:-1]    
    lbls = ['none'] + label.split('\t')

    # --- Store the labels and colors of the legend    
    geolegs[fname] = {'labels': lbls, 'colors':clrs}
    

pkl_fname = os.path.join(dirs.data, 'geotop_legends.pkl')

with open(pkl_fname, 'wb') as f:
    pickle.dump(geolegs, f)
print(f"geolegs pickled to file {os.path.basename(pkl_fname)}")
print(f"in directory: {dirs.data}")
    
geolegs


geolegs pickled to file geotop_legends.pkl
in directory: /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/


{'BRO GeoTOP Verticale doorsnede geologische eenheid 130173,479431.pdf': {'labels': ['none',
   'NUAAOP',
   'NUECgb',
   'NUEC1',
   'NUNIHO',
   'NUNIBA',
   'NUNBXWI-SI-KO',
   'NUBX',
   'NUKR-BXDE',
   'NUDR',
   'Nugs',
   'NUUR2',
   'NUST'],
  'colors': [array([205.,  92.,  92.]),
   [255.0, 255.0, 255.0],
   array([200., 200., 200.]),
   array([102., 205., 171.]),
   array([170., 255., 245.]),
   array([208., 130.,  40.]),
   array([152.,  47.,  10.]),
   array([255., 255.,  80.]),
   array([255., 235.,   0.]),
   array([176.,  48.,  96.]),
   array([255., 127.,  80.]),
   array([156., 156., 156.]),
   array([189., 183., 107.])]},
 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 126311,474187.pdf': {'labels': ['none',
   'a',
   'v',
   'k',
   'kz',
   'zf',
   'zm',
   'zg',
   'g',
   'she'],
  'colors': [array([ 95.,  95., 255.]),
   [255.0, 255.0, 255.0],
   array([200., 200., 200.]),
   array([157.,  78.,  64.]),
   array([  0., 146.,   0.]),
   array([

In [117]:
legend_colors

[[255.0, 255.0, 255.0],
 array([200., 200., 200.]),
 array([157.,  78.,  64.]),
 array([  0., 146.,   0.]),
 array([194., 207.,  92.]),
 array([255., 255.,   0.]),
 array([243., 225.,   6.]),
 array([231., 195.,  22.]),
 array([216., 163.,  32.]),
 array([ 95.,  95., 255.])]

# Retrieve the geotop legends (geolegs) from the pickled file

Instead of generating the legend colors clicking the legend boxes
on the image and the legend labels from the lists above, they
can be retrieved from te pickled file. 

In [108]:
# --- Retrieve the geoleg from the pickled file
pkl_fname = os.path.join(dirs.data, 'geotop_legends.pkl')

with open(pkl_fname, 'rb') as f:
    geolegs = pickle.load(f)

# Lithoklasses en Geologische Eenheden

Het Excelbestand "ARK-dwars-en-langsdsn-Geotop.xlsx" in de map dirs.data bevat alle geologische eenheden die voorkomen in de legenda's
van de 10 gedownloade geotop doorsneden in dirs.dino. Ik heb daar een horizontale en verticale doorlatendheid aan gekoppeld om mee te
kunnen modelleren. Dat heb ik ook gedaan voor de "meest waarschijnlijke lithoklassen" in de legenda van dezelfde doorsneden.

Ik heb ze hier overgenomen om er gemakkelijk gebruik van de te kunnen maken.



# Get the point data necessary to georeference the different cross sections

Now that we practiced sampling a Geotop X-ref image, we can do it for a larger set of geotop x-sections.

But before doing that, we will get the extent of each cross section in both pixel and real-world coordinates with z NAP and x measured along the cross section.

To get the extent of each x-section:

Each section is loaded in turn. For each section click on three points of the left z-axis
followed by three points on the bottom x-axis. The points to be clicked are the zero point
of the axis, the last tick of the axis and the bottom/end of the colored section, which
is the actual bottom and the actual end of the colored section. Together with the values
on the axes, to be read separately, the georeferencing can be uniquely done in terms
of the vertical NAP  coordinate and the horizontal coordinate along the section.

In [24]:
# --- Coordinate data for the various cross sections

# --- Dictionary to store the picked points (pixels)
geotopPtsData = {}

# --- Run over all the geotop files stored.
for geotop_pdf in glob(dirs.dino + '*.pdf'):
    
    # --- Use  basename as key.
    basename = os.path.basename(geotop_pdf)
    
    # --- Get the two pages of each geotop pdf as RGB image 
    geotop_xsec, geotop_leg = pdf2image.convert_from_path(geotop_pdf, dpi=300)
        
    geotop_xsec = np.asarray(geotop_xsec.convert("RGB"))  # map
    geotop_leg  = np.asarray(geotop_leg.convert("RGB"))  # legend (not used here) 
    
    # --- Instantiate the point picker with the map image
    picker = ImagePicker(geotop_xsec)
    
    # --- Click the 5 points
    points = picker.pick_points(n=-1, title=basename)

    # --- Store them
    geotopPtsData[basename] = points
    

##  Convert the points for each X-sec into a dict with keys denoting their meaning

For each geotop X-section, 6 coordinates have now been collected into the
geotopPtsData dictionary. The keys are the basenames of the geotop pdf files, and the values is a list of 5 clicked pixel coordinate pairs: 3 along the z axis, followed by
2 along the x-axes. (Because the first point on the x-axis is the same as the
third point of the z-axis, only 5 points have to be clicked).

The 3 clicked values along the z-axis are z=0, z=lowest tick and z=bottom of the X-section.
The 2 clicked values along the x-axis are x=largest tick and x=right end of the X-section

Next is to convert these 5 coordinate pairs into 10 numbers:
xp1, zp1, xp2, zp2, xp3, zp3, xp4, zp4, xp5, zp5

Then put them in a dictionary, with keys denoting their meaning.

This allows easy conversion into the columns of a pd.DataFrame with
basenames as index.

In [25]:
# --- The dictionary (copy to not destroy the captured points and use a shorter name)
db = geotopPtsData.copy()

# --- Convert the collected coordinate pairs into the z and x coordinates
for k in db.keys():
    
    # --- first the 5 coordinate pairs of each geotop X-section --> array
    pts = np.array(db[k])
    
    # --- Select the 3 z-values and the 3 x-values
    db[k] = {'px1': pts[0, 0], 'pz1': pts[0, 1],
             'px2': pts[1, 0], 'pz2': pts[1, 1],
             'px3': pts[2, 0], 'pz3': pts[2, 1],
             'px4': pts[3, 0], 'pz4': pts[3, 1],
             'px5': pts[4, 0], 'pz5': pts[4, 1]}

# --- Generate a DataFrame to hold the extents of all the geotop_pdf files
df_extents = pd.DataFrame(index=db.keys(),
                          columns=db[k].keys()
                          )

# --- Fill it with the captured points
for k in db.keys():   
    df_extents.loc[k] = db[k]

# --- Make sure their type is not object, but int.
for col in df_extents.columns:
    df_extents[col] = df_extents[col].astype(int)
    
# --- Show the DataFrame
df_extents

,px1,pz1,px2,pz2,px3,pz3,px4,pz4,px5,pz5
"BRO GeoTOP Verticale doorsnede geologische eenheid 130173,479431.pdf",366,400,366,1570,361,1705,2887,1705,2964,1705
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 126311,474187.pdf",364,400,364,1573,366,1703,2880,1703,2959,1705
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130049,479466.pdf",366,385,366,1565,366,1700,2827,1705,2969,1700
"BRO GeoTOP Verticale doorsnede geologische eenheid 129926,479462.pdf",330,373,332,1650,332,1703,2858,1705,2962,1703
"BRO GeoTOP Verticale doorsnede geologische eenheid 127484,477893.pdf",364,400,364,1570,364,1703,2803,1703,2959,1707
"BRO GeoTOP Verticale doorsnede geologische eenheid 130049,479466.pdf",364,385,366,1568,368,1703,2825,1703,2964,1703
"BRO GeoTOP Verticale doorsnede geologische eenheid 126311,474187.pdf",366,397,364,1568,364,1705,2880,1705,2959,1705
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 129926,479462.pdf",332,376,332,1647,330,1705,2856,1703,2962,1705
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 127484,477893.pdf",364,397,366,1568,366,1703,2801,1705,2962,1703
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130173,479431.pdf",366,397,364,1568,366,1703,2887,1707,2959,1707


## Add real world values of Pz1 (z1) and Pz2 (z2) as well as Px1 (x1) and Px2 (x2)

The world values of pixels Pz1, Pz2, Px1 and Px2 are provided by the user and are
given below. The values have been obtained by clicking the points using the
ImagePicker in a loop running over each geotop.pdf file in the dirs.dino directory.

When done, these two DataFrames are merged and the world_extent values z3 and x3 are
computed and then added to the DataFrame.

In [26]:
# --- Coordinates z1, z2, x1, x2 of the geotop cross sections
wrld_axes = pd.DataFrame(data=np.array([
                      [0.0, -45.0, -99.0, 0.0, 8400.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 5600.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 8250.0, 9999.0],
                      [0.0, -48.0, -99.0, 0.0, 7000.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 4800.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 8250.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 5600.0, 9999.0],
                      [0.0, -48.0, -99.0, 0.0, 7000.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 4800.0, 9999.0],
                      [0.0, -45.0, -99.0, 0.0, 8400.0, 9999.0]
                      ]),
             index=db.keys(),
             columns = ['z1', 'z2', 'z3', 'x1', 'x2', 'x3']
)

# --- Merge the two DataFrames
for col in wrld_axes.columns:
    df_extents[col] = wrld_axes[col]

# --- Show the extents DataFrame
df_extents

,px1,pz1,px2,pz2,px3,pz3,px4,pz4,px5,pz5,z1,z2,z3,x1,x2,x3
"BRO GeoTOP Verticale doorsnede geologische eenheid 130173,479431.pdf",366,400,366,1570,361,1705,2887,1705,2964,1705,0.0,-45.0,-99.0,0.0,8400.0,9999.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 126311,474187.pdf",364,400,364,1573,366,1703,2880,1703,2959,1705,0.0,-45.0,-99.0,0.0,5600.0,9999.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130049,479466.pdf",366,385,366,1565,366,1700,2827,1705,2969,1700,0.0,-45.0,-99.0,0.0,8250.0,9999.0
"BRO GeoTOP Verticale doorsnede geologische eenheid 129926,479462.pdf",330,373,332,1650,332,1703,2858,1705,2962,1703,0.0,-48.0,-99.0,0.0,7000.0,9999.0
"BRO GeoTOP Verticale doorsnede geologische eenheid 127484,477893.pdf",364,400,364,1570,364,1703,2803,1703,2959,1707,0.0,-45.0,-99.0,0.0,4800.0,9999.0
"BRO GeoTOP Verticale doorsnede geologische eenheid 130049,479466.pdf",364,385,366,1568,368,1703,2825,1703,2964,1703,0.0,-45.0,-99.0,0.0,8250.0,9999.0
"BRO GeoTOP Verticale doorsnede geologische eenheid 126311,474187.pdf",366,397,364,1568,364,1705,2880,1705,2959,1705,0.0,-45.0,-99.0,0.0,5600.0,9999.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 129926,479462.pdf",332,376,332,1647,330,1705,2856,1703,2962,1705,0.0,-48.0,-99.0,0.0,7000.0,9999.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 127484,477893.pdf",364,397,366,1568,366,1703,2801,1705,2962,1703,0.0,-45.0,-99.0,0.0,4800.0,9999.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130173,479431.pdf",366,397,364,1568,366,1703,2887,1707,2959,1707,0.0,-45.0,-99.0,0.0,8400.0,9999.0


In [29]:
# --- Compute the bottom coordinate of the z-axis
dfe = df_extents
dfe['z3'] = (dfe['z1'] + 
              (dfe['pz3'] - dfe['pz1']) / (dfe['pz2'] - dfe['pz1']) *
              (dfe['z2']  - dfe['z1'])
)

# --- Compute the largest coordinate of the x-axis
dfe['x3'] = (dfe['x1'] + 
              (dfe['px5'] - dfe['px3']) / (dfe['px4'] - dfe['px3']) *
              (dfe['x2']  - dfe['x1'])
)

# --- Just round for convenience
dfe['x3'] = np.round(dfe['x3'])

# --- Show the extents DataFrame, which is now complete
df_extents

,px1,pz1,px2,pz2,px3,pz3,px4,pz4,px5,pz5,z1,z2,z3,x1,x2,x3
"BRO GeoTOP Verticale doorsnede geologische eenheid 130173,479431.pdf",366,400,366,1570,361,1705,2887,1705,2964,1705,0.0,-45.0,-50.192308,0.0,8400.0,8656.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 126311,474187.pdf",364,400,364,1573,366,1703,2880,1703,2959,1705,0.0,-45.0,-49.987212,0.0,5600.0,5776.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130049,479466.pdf",366,385,366,1565,366,1700,2827,1705,2969,1700,0.0,-45.0,-50.148305,0.0,8250.0,8726.0
"BRO GeoTOP Verticale doorsnede geologische eenheid 129926,479462.pdf",330,373,332,1650,332,1703,2858,1705,2962,1703,0.0,-48.0,-49.992169,0.0,7000.0,7288.0
"BRO GeoTOP Verticale doorsnede geologische eenheid 127484,477893.pdf",364,400,364,1570,364,1703,2803,1703,2959,1707,0.0,-45.0,-50.115385,0.0,4800.0,5107.0
"BRO GeoTOP Verticale doorsnede geologische eenheid 130049,479466.pdf",364,385,366,1568,368,1703,2825,1703,2964,1703,0.0,-45.0,-50.135249,0.0,8250.0,8717.0
"BRO GeoTOP Verticale doorsnede geologische eenheid 126311,474187.pdf",366,397,364,1568,364,1705,2880,1705,2959,1705,0.0,-45.0,-50.264731,0.0,5600.0,5776.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 129926,479462.pdf",332,376,332,1647,330,1705,2856,1703,2962,1705,0.0,-48.0,-50.190401,0.0,7000.0,7294.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 127484,477893.pdf",364,397,366,1568,366,1703,2801,1705,2962,1703,0.0,-45.0,-50.187874,0.0,4800.0,5117.0
"BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130173,479431.pdf",366,397,364,1568,366,1703,2887,1707,2959,1707,0.0,-45.0,-50.187874,0.0,8400.0,8640.0


In [30]:
for fn in df_extents.index:
    rec = df_extents.loc[fn]
    print(rec['x1'], rec['x3'], rec['z1'], rec['z3'])
for fn in df_extents.index:
    rec = df_extents.loc[fn]
    print(rec['px3'], rec['px5'], rec['pz1'], rec['pz3'])

0.0 8656.0 0.0 -50.19230769230769
0.0 5776.0 0.0 -49.987212276214834
0.0 8726.0 0.0 -50.14830508474576
0.0 7288.0 0.0 -49.99216914643696
0.0 5107.0 0.0 -50.11538461538462
0.0 8717.0 0.0 -50.1352493660186
0.0 5776.0 0.0 -50.264730999146025
0.0 7294.0 0.0 -50.190401258851296
0.0 5117.0 0.0 -50.18787361229718
0.0 8640.0 0.0 -50.18787361229718
361.0 2964.0 400.0 1705.0
366.0 2959.0 400.0 1703.0
366.0 2969.0 385.0 1700.0
332.0 2962.0 373.0 1703.0
364.0 2959.0 400.0 1703.0
368.0 2964.0 385.0 1703.0
364.0 2959.0 397.0 1705.0
330.0 2962.0 376.0 1705.0
366.0 2962.0 397.0 1703.0
366.0 2959.0 397.0 1703.0


## Save the extents DataFrame to a pickle file for later retrieval

In [31]:
pkl_file = os.path.join(dirs.data, 'geotop_extents.pkl')

df_extents.to_pickle(pkl_file)

print(f"Geotop_pdf's extent saved to {pkl_file}")
print(f"That is to file {os.path.basename(pkl_file)}")
print(f"In directory {dirs.dino}")

Geotop_pdf's extent saved to /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/geotop_extents.pkl
That is to file geotop_extents.pkl
In directory /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/dinoloket/


# Retrieve the extents from the pickle file

Instead of gowing through the process to get the extents
of the geotop pdfs, the can be retrieved from the pickle file
directly

In [32]:
pkl_file = os.path.join(dirs.data, 'geotop_extents.pkl')

with open(pkl_file, 'rb') as f:
    df_extents = pickle.load(f)

## Sampling the geotop X-ssection voxel arrays

We are now ready to sample the cross section's voxels and match them with their
legend. For each geotop X-section file, we first click the extents of the
X-section and then match them with its world_extents, which is know from the
extents file.

The easiest way is to automatically fill the arrays, which is possible when we
store for each cross section the colors of the legend and the X-section
pixel-extent and world-extent.


In [ ]:
# --- Retrieve the geoleg from the pickled file
pkl_fname = os.path.join(dirs.data, 'geotop_legends.pkl')

with open(pkl_fname, 'rb') as f:
    geolegs = pickle.load(f)

# --- Retrieve the extents from the pickled file
pkl_file = os.path.join(dirs.data, 'geotop_extents.pkl')

with open(pkl_file, 'rb') as f:
    df_extents = pickle.load(f)

# ======== 

geotop_xsec = {}

# --- Run over all the geotop files stored.
for geotop_pdf in glob(dirs.dino + '*.pdf'):
    fname = os.path.basename(geotop_pdf)
    rec = df_extents.loc[fname]
    pxl_extent   = (rec['px1'], rec['px5'], rec['pz1'], rec['pz5'])
    world_extent = (rec['x1'],  rec['x3'],  rec['z3'],  rec['z1'])
    
    legend_colors = geolegs[fname]['colors']
    legend_labels = geolegs[fname]['labels']
        
    xsec_image = pdf2image.convert_from_path(geotop_pdf, dpi=300)[0]
    xsec_image = np.asarray(xsec_image.convert("RGB"))  
    
    digitizer = CrossSectionDigitizer(xsec_image)
    
    digitizer.legend_colors = legend_colors
    digitizer.set_pxl_bbox(*pxl_extent)
    digitizer.set_world_bbox(*world_extent)
    digitizer.set_grid(dx=100., dz=0.5)
    arr = digitizer.build_array()

    # --- We can uses this to store the entire array
    geotop_xsec[fname] = {'name': fname,
                          'pxl_extent': pxl_extent,
                          'world_extent': world_extent,
                          'colors': geolegs[fname]['colors'],
                          'labels': geolegs[fname]['labels'],
                          'shape': arr.shape,
                          'arr': arr,
                      }
    
    # plot_result(arr, world_extent=world_extent)
    show_filled_array(arr=arr, legend_colors=legend_colors)
    


-50.19230769230769 0.0
-49.987212276214834 0.0
-50.14830508474576 0.0
-49.99216914643696 0.0
-50.11538461538462 0.0
-50.1352493660186 0.0
-50.264730999146025 0.0
-50.190401258851296 0.0
-50.18787361229718 0.0
-50.18787361229718 0.0


# The geotop pdf file names split up in its parts for more easy use.

The x and y denote the beginning of each cross section.
The files are all BRO GeoTop vertical X-sections.
There are two type of files:
    Geological Unit
    Most probable litho-class

To reconstructie the file names just join the parts and add .pdf
The files are in dirs.dino

In [73]:
geotop_files = {
    0: {"type": "BRO GeoTOP Verticale doorsnede",
        "what": "geologische eenheid",
        "x": 130173, "y":479431},
    1: {"type": "BRO GeoTOP Verticale doorsnede",
        "what": "meest waarschijnlijke lithoklasse",
        "x": 126311,"y": 474187},
    2: {"type": "BRO GeoTOP Verticale doorsnede",
        "what": "meest waarschijnlijke lithoklasse",
        "x": 130049, "y": 479466},
    3: {"type": "BRO GeoTOP Verticale doorsnede",
        "what": "geologische eenheid",
        "x":129926, "y":479462},
    4: {"type": "BRO GeoTOP Verticale doorsnede",
        "what": "geologische eenheid",
        "x":127484, "y":477893},
    5: {"type": "BRO GeoTOP Verticale doorsnede",
        "what": "geologische eenheid",
        "x":130049, "y":479466},
    6: {"type": "BRO GeoTOP Verticale doorsnede",
        "what": "geologische eenheid",
        "x":126311, "y":474187},
    6: {"type": "BRO GeoTOP Verticale doorsnede",
        "what": "meest waarschijnlijke lithoklasse",
        "x": 129926,"y":479462},
    8: {"type": "BRO GeoTOP Verticale doorsnede",
        "what": "meest waarschijnlijke lithoklasse",
        "x": 127484,"y":477893},
    9: {"type": "BRO GeoTOP Verticale doorsnede",
        "what": "meest waarschijnlijke lithoklasse",
        "x": 130173,"y":479431},
}

In [85]:
for fnm, xsec in geotop_xsec.items():
    print(xsec.keys())
    break

dict_keys(['pxl_extent', 'world_extent', 'colors', 'labels', 'shape', 'idx'])


In [ ]:
def get_prop_arrays(xsec, geo_units=None):
    """Return the set of arrays with available properties defined in geo_units.
    
    Parameters
    ----------
    xsec: dictionary
        essential properties of the cross section.
        world_extent, pxl_extent, legend_labels
        kkh, kv, n, rhow
    geo_units: dictionary
        geo_units with properties
        keys are 0..n
        each key has properties unit, kh, kv, n, rho (dry), descr.
        
        The units can either by the dict of litho_classes or the dict of geo_units.
        Each covers properties of the x-section pertaining to the two used types of
        geotop x-section ("geologische eenheid", or "meest waarschijnljke litho_classe")
        
    Returns
    -------
    dictionary of arrays with the shape of xsec['arr']:
        {'kh':kh, 'kv':kv, 'n':n, 'rhow':rhow}
    
    """    
    legend_labels = xsec['labels']
    
    # --- Verify the geo_units match the legend_labels
    s = set(legend_labels).difference(geo_units.keys())
    
    assert len(s) == 0, "Legend_labels do not match geo_units. Maybe you passed the wrong geo_units dictionary"

    # --- Array with unit index or litho class index
    arr = xsec['arr']

    # --- Available properties defined for each unit or litho class
    kh   = np.zeros_like(arr)
    kv   = np.zeros_like(arr)
    n    = np.zeros_like(arr)
    rho  = np.zeros_like(arr)
    rhow = np.zeros_like(arr)
    
    # --- For each unit in the geo_units dict check if the item is there.
    for idx, label in enumerate(legend_labels):
        if label == 'none':
            continue
        mask = arr == idx
        kh[mask]   = geo_units[label]['kh']
        kv[mask]   = geo_units[label]['kv']
        n[mask]    = geo_units[label]['n']
        rho[mask]  = geo_units[label]['rho']
        
    rhow = n * 1000 + (1 - n) * rho

    return {'kh':kh, 'kv':kv, 'n':n, 'rhow':rhow}


Missing: NUECga
Missing: NUgs
Missing: NUST
Missing: NUST
Missing: NUST
Missing: NUST
Missing: NUST
Missing: NUST
Missing: NUST
Missing: NUST
Missing: NUST


## Fill model arrays

The "get_property_arrays" filled arrays of the shape equal to the voxel grid of the
cross section with the kh, kv, n, rho and rhow properties.

We can now sample these array with an arbitrary model grid.
Doing so, we have to deal with special zones, that are still empty or need a special treatment. For instance, water (boundary condition) and air (top of the model) and
things like pile sheetings and zones that need some other conductivity.

These zones are specified using a bounding box with their properties. The
same properties as we already have specified above.

Finally we need boundary conditions. These can be specified line-wise.

We also need to compute the over-pressure everywhere, for which we need the wet bulk density of all layers.

The total wet weight equals the cumsum of the wet weight from above downward, yielding
the value at the bottom of each cell. This has to match with the water pressure at the
same locations. This equals that at the centers of the cells and then taking the upward
flux into account compute the pressure at the bottom of these cells. Then compute the ratio of the water pressure over the total pressure and color that using safety-zone colors. This is doable.


In [ ]:
def patch_array(gr, arr, patch_extent, value):
    """Return patched array.
    
    Parameters
    ----------
    gr: mfgrid.Grid object
       Holds the modflow5 type grid.
    arr: np.ndarray
       array to be patched. Must have same shape as gr.
    patch_extent: 4 floats
       xmin, xmax, zmin, zmax of the patch
    value: float
       value to patch
    """
    pe = patch_extent
    pxmin, pxmax = min(pe[0], pe[1]), max(pe[0], pe[1])
    pzmin, pzmax = min(pe[2], pe[3]), max(pe[2], pe[3])

    x = gr.x
    z = gr.z

    ix1 = np.clip(np.searchsorted( x,  pxmin) - 1, 0, len(x) - 2)
    ix2 = np.clip(np.searchsorted( x,  pxmax) - 1, 0, len(x) - 2)
    iz1 = np.clip(np.searchsorted(-z, -pzmin) - 1, 0, len(z) - 2)
    iz2 = np.clip(np.searchsorted(-z, -pzmax) - 1, 0, len(z) - 2)
    
    arr[iz1:iz2, ix1:ix2] = value
    return arr

IndentationError: unindent does not match any outer indentation level (<string>, line 14)